In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = pd.read_csv('final_cleaned_data.csv', sep = ';')
df.head()

,Rank,Title,Artists,Date,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,...,# of Artist,Artist (Ind.),# of Nationality,Nationality,Continent,Points (Total),Points (Ind for each Artist/Nat),id,Song URL,Loudness_norm
0,1,Ella Baila Sola,"Eslabon Armado, Peso Pluma",2023-05-29,0.668,0.758,-5.176,0.033,0.483,0.000,...,Artist 1,Eslabon Armado,Nationality 1,Mexico,Latin-America,200,100.0,3qQbCzHBycnDpGskqOWY0E,https://open.spotify.com/track/3qQbCzHBycnDpGs...,0.849862
1,2,WHERE SHE GOES,Bad Bunny,2023-05-29,0.652,0.800,-4.019,0.061,0.143,0.629,...,Artist 1,Bad Bunny,Nationality 1,Puerto Rico,Latin-America,199,199.0,7ro0hRteUMfnOioTFI5TG1,https://open.spotify.com/track/7ro0hRteUMfnOio...,0.883423
2,3,La Bebe - Remix,"Yng Lvcas, Peso Pluma",2023-05-29,0.812,0.479,-5.678,0.333,0.213,0.000,...,Artist 1,Yng Lvcas,Nationality 1,Mexico,Latin-America,198,99.0,2UW7JaomAMuX9pZrjVpHAU,https://open.spotify.com/track/2UW7JaomAMuX9pZ...,0.835301
3,4,Cupid - Twin Ver.,FIFTY FIFTY,2023-05-29,0.783,0.592,-8.332,0.033,0.435,0.000,...,Artist 1,FIFTY FIFTY,Nationality 1,South Korea,Asia,197,197.0,7FbrGaHYVDmfr7KoLIZnQ7,https://open.spotify.com/track/7FbrGaHYVDmfr7K...,0.758318
4,5,un x100to,"Grupo Frontera, Bad Bunny",2023-05-29,0.569,0.724,-4.076,0.047,0.228,0.000,...,Artist 1,Grupo Frontera,Nationality 1,Mexico,Latin-America,196,98.0,6pD0ufEQq0xdHSsRbg9LBK,https://open.spotify.com/track/6pD0ufEQq0xdHSs...,0.881769


In [3]:
df.shape

(467061, 21)

In [4]:
df['Date'] = pd.to_datetime(df['Date'])

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 467061 entries, 0 to 467060
Data columns (total 21 columns):
 #   Column                            Non-Null Count   Dtype         
---  ------                            --------------   -----         
 0   Rank                              467061 non-null  int64         
 1   Title                             467061 non-null  object        
 2   Artists                           467061 non-null  object        
 3   Date                              467061 non-null  datetime64[ns]
 4   Danceability                      467061 non-null  float64       
 5   Energy                            467061 non-null  float64       
 6   Loudness                          467061 non-null  float64       
 7   Speechiness                       467061 non-null  float64       
 8   Acousticness                      467061 non-null  float64       
 9   Instrumentalness                  467061 non-null  float64       
 10  Valence                         

In [6]:
song_release_dates = df.groupby('id')['Date'].min()

In [7]:
df['first_appearance'] = df['id'].map(song_release_dates)

In [8]:
df.head()

,Rank,Title,Artists,Date,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,...,Artist (Ind.),# of Nationality,Nationality,Continent,Points (Total),Points (Ind for each Artist/Nat),id,Song URL,Loudness_norm,first_appearance
0,1,Ella Baila Sola,"Eslabon Armado, Peso Pluma",2023-05-29,0.668,0.758,-5.176,0.033,0.483,0.000,...,Eslabon Armado,Nationality 1,Mexico,Latin-America,200,100.0,3qQbCzHBycnDpGskqOWY0E,https://open.spotify.com/track/3qQbCzHBycnDpGs...,0.849862,2023-04-28
1,2,WHERE SHE GOES,Bad Bunny,2023-05-29,0.652,0.800,-4.019,0.061,0.143,0.629,...,Bad Bunny,Nationality 1,Puerto Rico,Latin-America,199,199.0,7ro0hRteUMfnOioTFI5TG1,https://open.spotify.com/track/7ro0hRteUMfnOio...,0.883423,2023-05-19
2,3,La Bebe - Remix,"Yng Lvcas, Peso Pluma",2023-05-29,0.812,0.479,-5.678,0.333,0.213,0.000,...,Yng Lvcas,Nationality 1,Mexico,Latin-America,198,99.0,2UW7JaomAMuX9pZrjVpHAU,https://open.spotify.com/track/2UW7JaomAMuX9pZ...,0.835301,2023-03-18
3,4,Cupid - Twin Ver.,FIFTY FIFTY,2023-05-29,0.783,0.592,-8.332,0.033,0.435,0.000,...,FIFTY FIFTY,Nationality 1,South Korea,Asia,197,197.0,7FbrGaHYVDmfr7KoLIZnQ7,https://open.spotify.com/track/7FbrGaHYVDmfr7K...,0.758318,2023-04-01
4,5,un x100to,"Grupo Frontera, Bad Bunny",2023-05-29,0.569,0.724,-4.076,0.047,0.228,0.000,...,Grupo Frontera,Nationality 1,Mexico,Latin-America,196,98.0,6pD0ufEQq0xdHSsRbg9LBK,https://open.spotify.com/track/6pD0ufEQq0xdHSs...,0.881769,2023-04-17


### Split train validation and test

In [9]:
release_year = song_release_dates.dt.year

def assign_split(year):
    if 2017 <= year <= 2020:
        return 'train'
    elif year == 2021:
        return 'val'
    elif 2022 <= year <= 2023:
        return 'test'
    else:
        return 'ignore' 

split_labels = release_year.apply(assign_split)


df['split'] = df['id'].map(split_labels)

In [10]:
df['split'].value_counts()

split
train    345628
val       61539
test      59894
Name: count, dtype: int64

In [11]:
train_start, train_end = '2017-01-01', '2020-12-31'
val_start, val_end     = '2021-01-01', '2021-12-31'
test_start, test_end   = '2022-01-01', '2023-12-31'


train_song_ids = song_release_dates[
    (song_release_dates >= train_start) & (song_release_dates <= train_end)
].index

val_song_ids = song_release_dates[
    (song_release_dates >= val_start) & (song_release_dates <= val_end)
].index

test_song_ids = song_release_dates[
    (song_release_dates >= test_start) & (song_release_dates <= test_end)
].index

In [12]:
train_df = df[df['id'].isin(train_song_ids)]
val_df   = df[df['id'].isin(val_song_ids)]
test_df  = df[df['id'].isin(test_song_ids)]

In [13]:
train_df['first_appearance'].describe()

count                           345628
mean     2018-09-30 21:16:41.293876736
min                2017-01-01 00:00:00
25%                2017-08-25 00:00:00
50%                2018-08-28 00:00:00
75%                2019-10-18 00:00:00
max                2020-12-31 00:00:00
Name: first_appearance, dtype: object

In [14]:
len(train_df), len(val_df), len(test_df)

(345628, 61539, 59894)

### Aggregation

In [15]:
song_perf_train = train_df.groupby('id').agg({
    'Rank': ['min', 'mean', 'count'],
    'Date': ['min', 'max'],
    'Title': 'first',
    'Artist (Ind.)': 'first'
}).reset_index()

song_perf_train.columns = ['id', 'best_rank', 'avg_rank', 'total_weeks_charted',
                            'first_appearance', 'last_appearance', 'Title', 'Artist']



In [16]:
song_perf_train.shape

(6158, 8)

In [17]:
print(f"  Unique songs: {len(song_perf_train):,}")
print(f"  Date range: {song_perf_train['first_appearance'].min().date()} to "
      f"{song_perf_train['first_appearance'].max().date()}")

  Unique songs: 6,158
  Date range: 2017-01-01 to 2020-12-31


In [18]:
song_perf_val = val_df.groupby('id').agg({
    'Rank': ['min', 'mean', 'count'],
    'Date': ['min', 'max'],
    'Title': 'first',
    'Artist (Ind.)': 'first'
}).reset_index()

song_perf_val.columns = ['id', 'best_rank', 'avg_rank', 'total_weeks_charted',
                           'first_appearance', 'last_appearance', 'Title', 'Artist']
song_perf_val.shape

(1414, 8)

In [19]:
song_perf_test = test_df.groupby('id').agg({
    'Rank': ['min', 'mean', 'count'],
    'Date': ['min', 'max'],
    'Title': 'first',
    'Artist (Ind.)': 'first'
}).reset_index()

song_perf_test.columns = ['id', 'best_rank', 'avg_rank', 'total_weeks_charted',
                           'first_appearance', 'last_appearance', 'Title', 'Artist']
song_perf_test.shape

(1589, 8)

### Check overlap 

In [20]:

train_ids = set(song_perf_train['id'])
val_ids   = set(song_perf_val['id'])
test_ids  = set(song_perf_test['id'])


overlap_train_val  = train_ids & val_ids
overlap_train_test = train_ids & test_ids
overlap_val_test   = val_ids & test_ids
        
print("train ∩ val:", len(overlap_train_val))
print("train ∩ test:", len(overlap_train_test))
print("val ∩ test:", len(overlap_val_test))

train ∩ val: 0
train ∩ test: 0
val ∩ test: 0


### Label popularity
- train

In [21]:
song_perf_train['peak_score'] = (201 - song_perf_train['best_rank']) / 200 * 100

# Calculate longevity score (cap at 20 weeks)
song_perf_train['longevity_score'] = np.minimum(song_perf_train['total_weeks_charted'] / 20, 1.0) * 100

# Calculate popularity score (weighted combination)
weight_peak = 0.4
weight_longevity = 0.6
song_perf_train['popularity_score'] = (
    weight_peak * song_perf_train['peak_score'] + 
    weight_longevity * song_perf_train['longevity_score']
)

train_threshold = song_perf_train['popularity_score'].quantile(0.7)
song_perf_train['popularity_label'] = song_perf_train['popularity_score'].apply(
    lambda x: 'Popular' if x >= train_threshold else 'Not Popular'
)
print(train_threshold)

83.77999999999993


- validation

In [22]:
song_perf_val['peak_score'] = (201 - song_perf_val['best_rank']) / 200 * 100

# Calculate longevity score (cap at 20 weeks)
song_perf_val['longevity_score'] = np.minimum(song_perf_val['total_weeks_charted'] / 20, 1.0) * 100


# Calculate popularity score (weighted combination)
weight_peak = 0.4
weight_longevity = 0.6
song_perf_val['popularity_score'] = (
    weight_peak * song_perf_val['peak_score'] + 
    weight_longevity * song_perf_val['longevity_score']
)
train_threshold = song_perf_train['popularity_score'].quantile(0.7)
song_perf_val['popularity_label'] = song_perf_val['popularity_score'].apply(
    lambda x: 'Popular' if x >= train_threshold else 'Not Popular'
)

- test

In [23]:
song_perf_test['peak_score'] = (201 - song_perf_test['best_rank']) / 200 * 100

# Calculate longevity score (cap at 20 weeks)
song_perf_test['longevity_score'] = np.minimum(song_perf_test['total_weeks_charted'] / 20, 1.0) * 100

# Calculate popularity score (weighted combination)
weight_peak = 0.4
weight_longevity = 0.6
song_perf_test['popularity_score'] = (
    weight_peak * song_perf_test['peak_score'] + 
    weight_longevity * song_perf_test['longevity_score']
)

train_threshold = song_perf_train['popularity_score'].quantile(0.7)
song_perf_test['popularity_label'] = song_perf_train['popularity_score'].apply(
    lambda x: 'Popular' if x >= train_threshold else 'Not Popular'
)

In [24]:
song_perf_test.head()

,id,best_rank,avg_rank,total_weeks_charted,first_appearance,last_appearance,Title,Artist,peak_score,longevity_score,popularity_score,popularity_label
0,00Ga884hbpVvCNyeQdle1U,99,149.000000,2,2023-03-10,2023-03-11,Violet Chemistry,Miley Cyrus,51.0,10.0,26.4,Popular
1,00I41xsW6SunZDJ5fB8KAd,65,65.000000,1,2022-07-15,2022-07-15,Safety Zone,j-hope,68.0,5.0,30.2,Not Popular
2,00imgaPlYRrMGn9o83hfmk,44,111.666667,3,2022-07-08,2022-07-10,LOOSE CHANGE,Brent Faiyaz,78.5,15.0,40.4,Not Popular
3,01VzuxoBYMAJLObfU0w6dI,195,195.000000,1,2022-07-08,2022-07-08,ADDICTIONS (FEAT. Tre' Amani),Brent Faiyaz,3.0,5.0,4.2,Popular
4,01kfSdF9zfcDLri5sSWEoL,173,183.500000,4,2022-09-10,2022-09-18,RAVE,Dxrk ãƒ€ãƒ¼ã‚¯,14.0,20.0,17.6,Not Popular


In [25]:
song_perf_train.head()

,id,best_rank,avg_rank,total_weeks_charted,first_appearance,last_appearance,Title,Artist,peak_score,longevity_score,popularity_score,popularity_label
0,000xQL6tZNLJzIrtIgxqSl,40,101.500000,118,2017-03-24,2017-09-12,Still Got Time (feat. PARTYNEXTDOOR),ZAYN,80.5,100.0,92.2,Popular
1,003VDDA7J3Xb2ZFlNx7nIZ,108,138.000000,2,2020-02-07,2020-02-08,YELL OH,Trippie Redd,46.5,10.0,24.6,Not Popular
2,003eoIwxETJujVWmNFMoZy,91,136.500000,14,2018-06-15,2018-06-28,Growing Pains,Alessia Cara,55.0,70.0,64.0,Not Popular
3,003vvx7Niy0yvhvHt4a68B,73,168.069643,560,2020-08-15,2023-01-01,Mr. Brightside,The Killers,64.0,100.0,85.6,Popular
4,00B7TZ0Xawar6NZ00JFomN,61,109.285714,14,2018-04-06,2018-04-19,Best Life (feat. Chance The Rapper),Cardi B,70.0,70.0,70.0,Not Popular


In [26]:
song_perf_val.head()

,id,best_rank,avg_rank,total_weeks_charted,first_appearance,last_appearance,Title,Artist,peak_score,longevity_score,popularity_score,popularity_label
0,00Blm7zeNqgYLPtW6zg8cj,7,70.861905,210,2021-11-05,2022-06-02,One Right Now (with The Weeknd),Post Malone,97.0,100.0,98.8,Popular
1,01BCvCKQNDmFQQQSjrzPnm,184,184.000000,1,2021-10-01,2021-10-01,GHIGLIOTTI - feat. Noyz rcos,Salmo,8.5,5.0,6.4,Not Popular
2,01K4zKU104LyJ8gMb7227B,20,79.187500,16,2021-11-12,2021-11-30,Nothing New (feat. Phoebe Bridgers) (Taylor's ...,Taylor Swift,90.5,80.0,84.2,Popular
3,01QdEx6kFr78ZejhQtWR5m,79,123.500000,2,2021-04-09,2021-04-10,Forever & Always (Piano Version) (Taylor's Ver...,Taylor Swift,61.0,10.0,30.4,Not Popular
4,01QhKtUc12FGw2wX7dMYvv,121,121.000000,1,2021-09-24,2021-09-24,LOCO - English Ver.,ITZY,40.0,5.0,19.0,Not Popular


In [27]:
print(f"\n  TRAINING LABELS (2017-2020):")
train_label_counts = song_perf_train['popularity_label'].value_counts()
print(f"  Popular:     {train_label_counts.get('Popular', 0):,} "
      f"({train_label_counts.get('Popular', 0)/len(song_perf_train)*100:.1f}%)")
print(f"  Not Popular: {train_label_counts.get('Not Popular', 0):,} "
      f"({train_label_counts.get('Not Popular', 0)/len(song_perf_train)*100:.1f}%)")

print(f"\n VALIDATION LABELS (2021-2023):")
val_label_counts = song_perf_val['popularity_label'].value_counts()
print(f"  Popular:     {val_label_counts.get('Popular', 0):,} "
      f"({val_label_counts.get('Popular', 0)/len(song_perf_val)*100:.1f}%)")
print(f"  Not Popular: {val_label_counts.get('Not Popular', 0):,} "
      f"({val_label_counts.get('Not Popular', 0)/len(song_perf_val)*100:.1f}%)")

print(f"\n TEST LABELS (2022-2023):")
test_label_counts = song_perf_test['popularity_label'].value_counts()
print(f"  Popular:     {test_label_counts.get('Popular', 0):,} "
      f"({test_label_counts.get('Popular', 0)/len(song_perf_test)*100:.1f}%)")
print(f"  Not Popular: {test_label_counts.get('Not Popular', 0):,} "
      f"({test_label_counts.get('Not Popular', 0)/len(song_perf_test)*100:.1f}%)")


  TRAINING LABELS (2017-2020):
  Popular:     1,848 (30.0%)
  Not Popular: 4,310 (70.0%)

 VALIDATION LABELS (2021-2023):
  Popular:     399 (28.2%)
  Not Popular: 1,015 (71.8%)

 TEST LABELS (2022-2023):
  Popular:     449 (28.3%)
  Not Popular: 1,140 (71.7%)


In [28]:
song_perf_train['split'] = 'train'
song_perf_val['split']   = 'val'
song_perf_test['split'] = 'test'


songs_labeled = pd.concat(
    [song_perf_train, song_perf_val, song_perf_test],
    ignore_index=True
)

### Add audio feature to aggregated dataset

Check unique audio data per song

In [29]:
features = ['Danceability','Energy','Loudness_norm','Speechiness',
            'Acousticness','Instrumentalness','Valence']
check = df.groupby('id')[features].nunique()
not_unique = check[(check > 1).any(axis=1)]
print(f'IDs with non-unique feature values: {len(not_unique)}')

IDs with non-unique feature values: 0


In [30]:
features = ['Danceability','Energy','Loudness_norm','Speechiness',
            'Acousticness','Instrumentalness','Valence']
song_features = (
    df.sort_values(['id', 'Date'])
      .drop_duplicates('id', keep='first')[['id'] + features]
)
song_features

,id,Danceability,Energy,Loudness_norm,Speechiness,Acousticness,Instrumentalness,Valence
406858,000xQL6tZNLJzIrtIgxqSl,0.748,0.627,0.825120,0.064,0.131,0.0,0.524
447180,003VDDA7J3Xb2ZFlNx7nIZ,0.842,0.578,0.824511,0.138,0.004,0.0,0.190
325270,003eoIwxETJujVWmNFMoZy,0.353,0.755,0.817955,0.733,0.082,0.0,0.437
443397,003vvx7Niy0yvhvHt4a68B,0.352,0.911,0.848296,0.075,0.001,0.0,0.236
338175,00B7TZ0Xawar6NZ00JFomN,0.620,0.625,0.784249,0.553,0.287,0.0,0.665
...,...,...,...,...,...,...,...,...
104934,7zjEyeBsaw9gV0jofJLfOM,0.767,0.313,0.650210,0.080,0.838,0.0,0.765
129874,7zl7kehxesNEo2pYkKXTSe,0.924,0.730,0.822335,0.274,0.053,0.0,0.848
450034,7znO2T2deQ7nZUbyxEAMDB,0.570,0.535,0.803626,0.212,0.264,0.0,0.073
67100,7zvfDihYiJ8RQ1nRcpKBF5,0.727,0.530,0.750080,0.312,0.283,0.0,0.258


merge audio feature to aggregated data

In [31]:
songs_labeled = songs_labeled.merge(song_features, on='id', how='left')
songs_labeled

,id,best_rank,avg_rank,total_weeks_charted,first_appearance,last_appearance,Title,Artist,peak_score,longevity_score,popularity_score,popularity_label,split,Danceability,Energy,Loudness_norm,Speechiness,Acousticness,Instrumentalness,Valence
0,000xQL6tZNLJzIrtIgxqSl,40,101.500000,118,2017-03-24,2017-09-12,Still Got Time (feat. PARTYNEXTDOOR),ZAYN,80.5,100.0,92.2,Popular,train,0.748,0.627,0.825120,0.064,0.131,0.000,0.524
1,003VDDA7J3Xb2ZFlNx7nIZ,108,138.000000,2,2020-02-07,2020-02-08,YELL OH,Trippie Redd,46.5,10.0,24.6,Not Popular,train,0.842,0.578,0.824511,0.138,0.004,0.000,0.190
2,003eoIwxETJujVWmNFMoZy,91,136.500000,14,2018-06-15,2018-06-28,Growing Pains,Alessia Cara,55.0,70.0,64.0,Not Popular,train,0.353,0.755,0.817955,0.733,0.082,0.000,0.437
3,003vvx7Niy0yvhvHt4a68B,73,168.069643,560,2020-08-15,2023-01-01,Mr. Brightside,The Killers,64.0,100.0,85.6,Popular,train,0.352,0.911,0.848296,0.075,0.001,0.000,0.236
4,00B7TZ0Xawar6NZ00JFomN,61,109.285714,14,2018-04-06,2018-04-19,Best Life (feat. Chance The Rapper),Cardi B,70.0,70.0,70.0,Not Popular,train,0.620,0.625,0.784249,0.553,0.287,0.000,0.665
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9156,7ySUcLPVX7KudhnmNcgY2D,80,131.000000,4,2023-02-13,2023-02-16,S&M,Rihanna,60.5,20.0,36.2,Popular,test,0.769,0.684,0.854416,0.041,0.011,0.000,0.835
9157,7z84Fwf1R3Z2BwHCP620CI,94,130.000000,3,2022-09-29,2022-10-01,This Is Why,Paramore,53.5,15.0,30.4,Popular,test,0.723,0.737,0.780537,0.039,0.004,0.003,0.689
9158,7zBscbZUCr4jEABrfV9g03,70,94.750000,4,2022-11-04,2022-11-07,Before The Day Is Over,Joji,65.5,20.0,38.2,Popular,test,0.393,0.305,0.670138,0.046,0.765,0.001,0.150
9159,7zcJSflhqcSGICHUjgncnj,13,13.000000,1,2022-10-21,2022-10-21,Mastermind,Taylor Swift,94.0,5.0,40.6,Not Popular,test,0.661,0.354,0.591356,0.142,0.549,0.001,0.123


### Save aggregate dataset

In [32]:
output_file = 'aggregated_dataset.csv'
songs_labeled.to_csv(output_file, index=False, sep=';')
print(f" Saved to: {output_file}")
print(f"  Shape: {songs_labeled.shape}")

 Saved to: aggregated_dataset.csv
  Shape: (9161, 20)


In [33]:
df['split'].value_counts()

split
train    345628
val       61539
test      59894
Name: count, dtype: int64

In [34]:
df.head()

,Rank,Title,Artists,Date,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,...,# of Nationality,Nationality,Continent,Points (Total),Points (Ind for each Artist/Nat),id,Song URL,Loudness_norm,first_appearance,split
0,1,Ella Baila Sola,"Eslabon Armado, Peso Pluma",2023-05-29,0.668,0.758,-5.176,0.033,0.483,0.000,...,Nationality 1,Mexico,Latin-America,200,100.0,3qQbCzHBycnDpGskqOWY0E,https://open.spotify.com/track/3qQbCzHBycnDpGs...,0.849862,2023-04-28,test
1,2,WHERE SHE GOES,Bad Bunny,2023-05-29,0.652,0.800,-4.019,0.061,0.143,0.629,...,Nationality 1,Puerto Rico,Latin-America,199,199.0,7ro0hRteUMfnOioTFI5TG1,https://open.spotify.com/track/7ro0hRteUMfnOio...,0.883423,2023-05-19,test
2,3,La Bebe - Remix,"Yng Lvcas, Peso Pluma",2023-05-29,0.812,0.479,-5.678,0.333,0.213,0.000,...,Nationality 1,Mexico,Latin-America,198,99.0,2UW7JaomAMuX9pZrjVpHAU,https://open.spotify.com/track/2UW7JaomAMuX9pZ...,0.835301,2023-03-18,test
3,4,Cupid - Twin Ver.,FIFTY FIFTY,2023-05-29,0.783,0.592,-8.332,0.033,0.435,0.000,...,Nationality 1,South Korea,Asia,197,197.0,7FbrGaHYVDmfr7KoLIZnQ7,https://open.spotify.com/track/7FbrGaHYVDmfr7K...,0.758318,2023-04-01,test
4,5,un x100to,"Grupo Frontera, Bad Bunny",2023-05-29,0.569,0.724,-4.076,0.047,0.228,0.000,...,Nationality 1,Mexico,Latin-America,196,98.0,6pD0ufEQq0xdHSsRbg9LBK,https://open.spotify.com/track/6pD0ufEQq0xdHSs...,0.881769,2023-04-17,test


### Save split cleaned dataset

In [35]:
output_file = 'split_cleaned_dataset.csv'
df.to_csv(output_file, index=False, sep=';')
print(f" Saved to: {output_file}")
print(f" Shape: {df.shape}")

 Saved to: split_cleaned_dataset.csv
 Shape: (467061, 23)


In [38]:
df.head(10)

,Rank,Title,Artists,Date,Danceability,Energy,Loudness,Speechiness,Acousticness,Instrumentalness,...,# of Nationality,Nationality,Continent,Points (Total),Points (Ind for each Artist/Nat),id,Song URL,Loudness_norm,first_appearance,split
0,1,Ella Baila Sola,"Eslabon Armado, Peso Pluma",2023-05-29,0.668,0.758,-5.176,0.033,0.483,0.000,...,Nationality 1,Mexico,Latin-America,200,100.0,3qQbCzHBycnDpGskqOWY0E,https://open.spotify.com/track/3qQbCzHBycnDpGs...,0.849862,2023-04-28,test
1,2,WHERE SHE GOES,Bad Bunny,2023-05-29,0.652,0.800,-4.019,0.061,0.143,0.629,...,Nationality 1,Puerto Rico,Latin-America,199,199.0,7ro0hRteUMfnOioTFI5TG1,https://open.spotify.com/track/7ro0hRteUMfnOio...,0.883423,2023-05-19,test
2,3,La Bebe - Remix,"Yng Lvcas, Peso Pluma",2023-05-29,0.812,0.479,-5.678,0.333,0.213,0.000,...,Nationality 1,Mexico,Latin-America,198,99.0,2UW7JaomAMuX9pZrjVpHAU,https://open.spotify.com/track/2UW7JaomAMuX9pZ...,0.835301,2023-03-18,test
3,4,Cupid - Twin Ver.,FIFTY FIFTY,2023-05-29,0.783,0.592,-8.332,0.033,0.435,0.000,...,Nationality 1,South Korea,Asia,197,197.0,7FbrGaHYVDmfr7KoLIZnQ7,https://open.spotify.com/track/7FbrGaHYVDmfr7K...,0.758318,2023-04-01,test
4,5,un x100to,"Grupo Frontera, Bad Bunny",2023-05-29,0.569,0.724,-4.076,0.047,0.228,0.000,...,Nationality 1,Mexico,Latin-America,196,98.0,6pD0ufEQq0xdHSsRbg9LBK,https://open.spotify.com/track/6pD0ufEQq0xdHSs...,0.881769,2023-04-17,test
5,6,Flowers,Miley Cyrus,2023-05-29,0.707,0.681,-4.325,0.067,0.063,0.000,...,Nationality 1,United States,Anglo-America,195,195.0,4DHcnVTT87F0zZhRPYmZ3B,https://open.spotify.com/track/4DHcnVTT87F0zZh...,0.874547,2023-03-10,test
6,7,Daylight,David Kushner,2023-05-29,0.508,0.430,-9.475,0.034,0.830,0.000,...,Nationality 1,United States,Anglo-America,194,194.0,1odExI7RdWc4BT515LTAwj,https://open.spotify.com/track/1odExI7RdWc4BT5...,0.725163,2023-04-14,test
7,8,Kill Bill,SZA,2023-05-29,0.644,0.735,-5.747,0.039,0.052,0.144,...,Nationality 1,United States,Anglo-America,193,193.0,1Qrg8KqiBpW07V7PNxwwwL,https://open.spotify.com/track/1Qrg8KqiBpW07V7...,0.833299,2022-12-09,test
8,9,As It Was,Harry Styles,2023-05-29,0.520,0.731,-5.338,0.056,0.342,0.001,...,Nationality 1,United Kingdom,Europe,192,192.0,4Dvkj6JhhA12EX05fT7y2e,https://open.spotify.com/track/4Dvkj6JhhA12EX0...,0.845163,2022-05-20,test
9,10,TQG,"KAROL G, Shakira",2023-05-29,0.720,0.630,-3.547,0.277,0.673,0.000,...,Nationality 1,Colombia,Latin-America,191,95.5,0DWdj2oZMBFSzRsi2Cvfzf,https://open.spotify.com/track/0DWdj2oZMBFSzRs...,0.897114,2023-02-24,test
